# Global Variables

Global variables are team-scoped key/value pairs injected into workflow state at runtime. At
execution time they are merged into the run's **dynamic variables**, so a node references them by
their bare name with the `{{<name>}}` template syntax (e.g. `{{company_name}}`). Values can be
plain or **secret** (encrypted at rest, redacted in API responses). This notebook covers the full
lifecycle via `client.global_variables`: create, bulk-create, list, get, update, delete, and
resolve — and ends with an end-to-end workflow run that shows the substitution in action.

Everything created here is deleted in the cleanup cell.


> **These notebooks are async-first.** They use `AsyncWorkflowClient` with top-level `await`,
> which runs directly in Jupyter (the setup cell calls `nest_asyncio.apply()`). Every method
> shown also exists on the synchronous `WorkflowClient` — just drop the `await`. See the
> [docs](../docs/README.md) for the sync surface. Notebook bodies stay 100% async — there is
> no per-notebook sync cell.

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import os, sys
import nest_asyncio
from dotenv import load_dotenv
from pathlib import Path

# This is required to run asyncio in Jupyter Notebook
nest_asyncio.apply()

# Get the current notebook directory and find the project root
current_dir = Path(os.getcwd())
project_root = current_dir
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
else:
    project_root = current_dir
    for _ in range(5):
        if (project_root / 'pyproject.toml').exists():
            break
        project_root = project_root.parent

# Load the environment variables from .env in project root
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path), override=True)
    print(f"Loaded .env from: {env_path}")
else:
    print(f"Warning: .env file not found at {env_path}")

# Add the project root to sys.path so imports resolve
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Added to Python path: {project_root}")
else:
    print(f"Project root already in Python path: {project_root}")

In [ ]:
# Interactly credentials are read from environment variables.
#
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell
# environment always wins (setdefault only fills in what you have not set).
os.environ.setdefault("INTERACTLY_BASE_URL", "https://api-dev.interactly.ai/workflows")
os.environ.setdefault("INTERACTLY_TEAM_ID", "67458e762b7d3dc15aaea5b5")
os.environ.setdefault("INTERACTLY_USER_ID", "687b1a4f745c8e6806c98d91")

# The bearer token is a secret — never hardcode it in the notebook.
# Export it before launching Jupyter:  export INTERACTLY_API_KEY="…"
assert os.environ.get("INTERACTLY_API_KEY"), (
    "Set INTERACTLY_API_KEY in your environment before running this notebook."
)

#print(f"API KEY is: {os.getenv('INTERACTLY_API_KEY')}")
print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"USER ID is: {os.getenv('INTERACTLY_USER_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import json

from interactly import AsyncWorkflowClient

In [ ]:
client = AsyncWorkflowClient()
print("Connected to", client._base_url)

In [ ]:
# Idempotency guard — remove any variables left over from a previous run so that
# re-running this notebook always starts from a clean slate (names must be unique).
_demo_names = {"company_name", "partner_api_key", "support_email", "support_phone", "business_hours"}
_existing = await (await client.global_variables.list(size=200)).list_all()
for _v in _existing:
    if _v.name in _demo_names:
        await client.global_variables.delete(_v.id)
print("Cleared any pre-existing demo variables.")


## 1. Create a single variable

`create()` needs a unique `name`; `value`, `description`, and `category` are optional. Set
`is_secret=True` to encrypt the value at rest — it will be redacted when you read it back.

In [ ]:
greeting = await client.global_variables.create(
    name="company_name",
    value="Interactly",
    description="Displayed in the agent greeting.",
    category="branding",
)
GREETING_ID = greeting.id
print(f"Created {greeting.name!r}  id={GREETING_ID}")

api_key = await client.global_variables.create(
    name="partner_api_key",
    value="sk-super-secret-value",
    category="secrets",
    is_secret=True,
)
SECRET_ID = api_key.id
print(f"Created secret {api_key.name!r}  is_secret={api_key.is_secret}")

## 2. Bulk-create

`bulk_create()` takes a list of variable dicts and creates them in one request. It returns the
successfully created variables and raises `ValueError` if the server reports per-item errors.

In [ ]:
bulk = await client.global_variables.bulk_create(
    variables=[
        {"name": "support_email", "value": "help@interactly.ai", "category": "contact"},
        {"name": "support_phone", "value": "+1-555-0100", "category": "contact"},
        {"name": "business_hours", "value": "9am-5pm ET", "category": "contact"},
    ]
)
BULK_IDS = [v.id for v in bulk]
print(f"Bulk-created {len(bulk)} variable(s): {[v.name for v in bulk]}")

## 3. List, filter, and fetch

`list()` is paginated (`page`/`size`) with `search` and `category` filters. `get()` fetches one
by ID — a secret's `value` comes back redacted.

In [ ]:
contact_vars = await client.global_variables.list(category="contact")
print(f"{contact_vars.total} variable(s) in the 'contact' category")
for v in contact_vars.items:
    print(f"  {v.name} = {v.value!r}")

fetched_secret = await client.global_variables.get(SECRET_ID)
print(f"Secret {fetched_secret.name!r} value (redacted): {fetched_secret.value!r}")

## 4. Update a variable

`update()` only sends the fields you pass (others stay untouched via the `NOT_GIVEN` sentinel).

In [ ]:
updated = await client.global_variables.update(
    GREETING_ID,
    value="Interactly AI",
    description="Brand name used in greetings and sign-offs.",
)
print(f"Updated value: {updated.value!r}")

## 5. Resolve everything for execution

`resolve()` returns a flat `{name: value}` dict with **secrets decrypted server-side**. This is
what execution services use to inject variables into workflow state. Handle the result carefully
— it contains plaintext secrets.

In [ ]:
resolved = await client.global_variables.resolve()
print(f"Resolved {len(resolved)} variable(s).")
# Show the non-secret ones we created. In a workflow these are referenced by bare name.
for name in ("company_name", "support_email", "support_phone"):
    if name in resolved:
        print(f"  {{{{{name}}}}} -> {resolved[name]!r}")


## 6. Using variables in a workflow

Inside any node config that accepts templated text (prompts, static messages, request bodies),
reference a global by its bare name with `{{<name>}}`. At execution time the team's globals are
resolved and merged into the run's **dynamic variables**, so a placeholder like `{{company_name}}`
is substituted before the node executes — exactly like any other dynamic variable. For example, a
greeting prompt might read:

```
Greet the caller on behalf of {{company_name}}.
If they need a human, share {{support_email}} and {{support_phone}}.
```

No SDK call is needed for substitution — it happens during execution once the variables exist.


### Run it end-to-end

The snippets above are just templates — let's prove the substitution actually happens. Below we
build a **single-node workflow** whose one static "say" node greets the caller using two of the
globals we created, execute a run, and inspect the emitted message.

At execution time the team's global variables are resolved and merged into the run's **dynamic
variables**, so a node references them by their bare name — `{{company_name}}`,
`{{support_email}}` — exactly like any other dynamic variable. Because `company_name` was updated
to `"Interactly AI"` in section 4, that is the value that appears at runtime. The demo workflow is
deleted in the cleanup cell.


In [ ]:
from interactly.configs import (
    SayStaticMessageNodeConfig,
    StaticMessagesConfig,
    WorkflowConfig,
    WorkflowConfigFullyHydrated,
)
from interactly.types.workflows.workflow import Workflow

# A single static "say" node (also the start node) whose message references two of the
# globals created above. At runtime the team's globals are merged into the run's dynamic
# variables, so they are substituted with the bare-name syntax: {{company_name}}.
say_node = SayStaticMessageNodeConfig(
    name="Global Greeting",
    is_start=True,
    static_messages_config=StaticMessagesConfig(
        static_messages=[
            "Hello! Thanks for calling {{company_name}}. "
            "You can always reach us at {{support_email}}."
        ]
    ),
)

demo_config = WorkflowConfigFullyHydrated(
    workflow_config=WorkflowConfig(name="Global Variables Demo", category="SDK Examples"),
    node_configs=[say_node],
    edge_configs=[],
)

demo_workflow: Workflow = await client.workflows.create_from_config(demo_config)
DEMO_WORKFLOW_ID = demo_workflow.id
print(f"Created workflow  id={DEMO_WORKFLOW_ID}  name={demo_workflow.name!r}")


In [ ]:
from interactly import WorkflowCommand
from interactly.runtime.events import SayStaticMessageNodeEvent, parse_event
from interactly.types.runs.interactive_run import InteractiveRunResponse

# Execute a single START turn — the say-static node fires immediately and ends the run.
response: InteractiveRunResponse = await client.runs.execute(
    workflow_id=DEMO_WORKFLOW_ID,
    command=WorkflowCommand.START,
)
print(f"Run ended — status: {response.status}  ({len(response.events)} events)\n")


def _text(message):
    """The event `message` may be a plain string, a dict, or a message object."""
    if isinstance(message, dict):
        return message.get("content", message)
    return getattr(message, "content", message)


# The say-static node's emitted message has the {{...}} placeholders resolved from the globals.
for event in response.events:
    typed = parse_event(event)
    if isinstance(typed, SayStaticMessageNodeEvent):
        print("Say-static node emitted (globals resolved):")
        print(f"  {_text(typed.message)}")


## 7. Cleanup

Delete every variable created in this notebook.

In [ ]:
for var_id in [GREETING_ID, SECRET_ID, *BULK_IDS]:
    await client.global_variables.delete(var_id)
print(f"Deleted {2 + len(BULK_IDS)} variable(s).")

# Delete the demo workflow created in section 6 (if that section was run).
try:
    await client.workflows.delete(DEMO_WORKFLOW_ID)
    print(f"Deleted workflow {DEMO_WORKFLOW_ID}.")
except NameError:
    pass  # section 6 end-to-end demo was not run

await client.close()


## See also

- Guide: [`../docs/guides/global_variables.md`](../docs/guides/global_variables.md)
- [`02_build_a_workflow.ipynb`](02_build_a_workflow.ipynb) — where `{{<name>}}` variables get used in node configs
- [`04_pagination_and_filtering.ipynb`](04_pagination_and_filtering.ipynb) — paging and filtering `list()`
- [`06_interactly_configs.ipynb`](06_interactly_configs.ipynb) — type-safe config authoring
